# Patcher Node — Unit Tests

Exercises `patcher_node` on real data: loads the rules_bank + legacy +
spec, runs Loader → Planner once, then runs the Patcher on the FIRST
operation of the resulting plan.

Each Patcher call is one LLM invocation. A full plan has ~25 ops, so
running the Patcher on the whole plan would cost roughly 25 LLM calls —
this notebook deliberately stops after a few to keep the cost low.

**Pre-requisites:**
1. `data/inputs/3gpp/28532-i00.md`
2. `data/inputs/rules/rules_bank_*.json`
3. `data/inputs/legacy/TS28532_ProvMnS.yaml` (optional)
4. `OPENAI_API_KEY` in `.env`
5. Optional: a Qdrant collection for RAG; if absent the Patcher degrades cleanly.

In [7]:
# Step 1 — Imports + paths
#
# Test fixtures (which spec, rules, legacy to use) live in
# openapi_generator.config.paths, so the notebook stays free of glob magic
# and ad-hoc strings. Edit paths.py to switch fixtures.

import json
from pathlib import Path

import yaml

from openapi_generator.config import get_logger
from openapi_generator.config.paths import (
    ROOT,
    TEST_LEGACY_PATH,
    TEST_RULES_PATH,
    TEST_SPEC_PATH,
)
from openapi_generator.nodes.loader import loader_node
from openapi_generator.nodes.planner import planner_node
from openapi_generator.nodes.patcher import patcher_node

logger = get_logger(__name__)

REPO_ROOT   = Path(ROOT).resolve()
SPEC_PATH   = Path(TEST_SPEC_PATH).resolve()
RULES_PATH  = Path(TEST_RULES_PATH).resolve()
LEGACY_PATH = Path(TEST_LEGACY_PATH).resolve()

assert SPEC_PATH.is_file(),  f"Spec not found at {SPEC_PATH}"
assert RULES_PATH.is_file(), f"Rules not found at {RULES_PATH}"
# Legacy is optional — Patcher must work without it too.
legacy_present = LEGACY_PATH.is_file()

logger.info(f"Spec   : {SPEC_PATH.name}")
logger.info(f"Rules  : {RULES_PATH.name}")
logger.info(f"Legacy : {LEGACY_PATH.name if legacy_present else '(none)'}")

2026-05-26 16:50:35 [INFO] __main__: Spec   : 28532-i00.md
2026-05-26 16:50:35 [INFO] __main__: Rules  : rules_bank_28532-i00_full_20260427_214528.json
2026-05-26 16:50:35 [INFO] __main__: Legacy : rel_17_TS28532_ProvMnS.yaml


In [8]:
# Step 2 — Build a realistic state with Loader + Planner first

with open(RULES_PATH, "r", encoding="utf-8") as f:
    rules_bank = json.load(f)

legacy_openapi = None
if legacy_present:
    with open(LEGACY_PATH, "r", encoding="utf-8") as f:
        legacy_openapi = yaml.safe_load(f)

loader_state = {
    "spec_doc_path": str(SPEC_PATH),
    "rules_bank": rules_bank,
    "legacy_openapi": legacy_openapi,
}
loader_out = loader_node(loader_state)
planner_out = planner_node({**loader_state, **loader_out})
ops = planner_out["operations_plan"]

logger.info(f"Planner produced {len(ops)} operation(s).")
for i, op in enumerate(ops[:5]):
    logger.info(
        f"  {i}. [{op['priority']:>6}] {op['action']:>6} {op['method'].upper():>6} "
        f"{op['path']}  (rules={op['source_rule_ids']})"
    )

2026-05-26 16:50:35 [INFO] openapi_generator.nodes.loader: Loader → parsed 696 sections from 28532-i00.md (excluded 1 symbolic-title section(s))
2026-05-26 16:50:35 [INFO] openapi_generator.nodes.loader: Loader → rules_bank: 240 rule(s); legacy_openapi: present
2026-05-26 16:50:35 [INFO] openapi_generator.nodes.loader: Loader → seeding final_openapi from legacy (1 path(s), 16 schema(s))
2026-05-26 16:50:35 [INFO] openapi_generator.nodes.planner: Planner Node started.
2026-05-26 16:51:05 [INFO] openapi_generator.nodes.planner: Planner Node complete — 25 operation(s): 21 create / 4 update / 0 keep; rules covered: 219/240
2026-05-26 16:51:05 [INFO] __main__: Planner produced 25 operation(s).
2026-05-26 16:51:05 [INFO] __main__:   0. [  high] update    PUT /{className}={id}  (rules=[0, 1, 2, 3, 4, 5, 9, 10, 11, 12, 13, 17, 18, 21, 22, 23, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 209, 210, 211, 212, 213, 214, 215

## Scenario A — first operation of the real plan

Run the Patcher on `operations_plan[0]`. One LLM call. RAG may or may not return chunks depending on whether Qdrant is up — the Patcher tolerates both.

In [9]:
# Step 3 — Compose Patcher state and invoke

state = {
    **loader_state,
    **loader_out,
    "operations_plan": ops,
    "current_op_idx": 0,
    "op_iteration_count": 0,
}
out = patcher_node(state)
frag = out["current_fragment"]

assert out["op_iteration_count"] == 1
assert frag["path"] == ops[0]["path"]
assert frag["method"] == ops[0]["method"]
logger.info(f"Fragment for {frag['method'].upper()} {frag['path']}")
logger.info(f"  paths      : {list((frag.get('paths') or {}).keys())}")
logger.info(
    f"  new schemas: {list(((frag.get('components') or {}).get('schemas') or {}).keys())}"
)

2026-05-26 16:51:05 [INFO] openapi_generator.nodes.patcher: Patcher → op 1/25 (PUT /{className}={id}) action=update attempt=1
2026-05-26 16:51:05 [WARNING] openapi_generator.rag.retriever: RAG query failed (ModuleNotFoundError: No module named 'langchain.retrievers') — returning []
2026-05-26 16:51:20 [INFO] openapi_generator.nodes.patcher: Patcher → produced fragment for PUT /{className}={id}: 1 path block(s), 0 new schema(s)
2026-05-26 16:51:20 [INFO] __main__: Fragment for PUT /{className}={id}
2026-05-26 16:51:20 [INFO] __main__:   paths      : ['/{className}={id}']
2026-05-26 16:51:20 [INFO] __main__:   new schemas: []


In [10]:
# Step 4 — Structural sanity on the fragment
#  - paths[path][method] exists and is a dict
#  - operation has summary + responses
#  - every {var} in the path appears as a path parameter

import re

path, method = frag["path"], frag["method"]
paths_block = frag.get("paths") or {}
assert path in paths_block, f"path {path!r} missing from fragment.paths"
op_obj = paths_block[path].get(method)
assert isinstance(op_obj, dict), f"missing operation object at paths.{path}.{method}"

assert "responses" in op_obj, "operation must define responses"
assert any(code.startswith("2") or code == "default" for code in op_obj["responses"]), \
    "operation must define at least one success/default response"

path_vars = set(re.findall(r"\{([^}]+)\}", path))
declared = {
    p["name"]
    for p in (op_obj.get("parameters") or [])
    if p.get("in") == "path"
}
missing = path_vars - declared
assert not missing, f"path variables not declared as parameters: {missing}"
logger.info(
    f"Sanity OK: responses={list(op_obj['responses'])}, "
    f"path_vars={sorted(path_vars)}, declared={sorted(declared)}"
)

2026-05-26 16:51:20 [INFO] __main__: Sanity OK: responses=['200'], path_vars=['className', 'id'], declared=['className', 'id']


In [11]:
# Step 5 — Inspect the produced YAML

print(yaml.safe_dump(
    {"paths": paths_block, "components": frag.get("components") or {}},
    sort_keys=False, allow_unicode=True,
))

paths:
  /{className}={id}:
    put:
      summary: Replaces a complete single resource or creates it if it does not exist
      description: With HTTP PUT a complete resource is replaced or created if it
        does not exist. The target resource is identified by the target URI.
      operationId: putResourceByClassNameAndId
      parameters:
      - name: className
        in: path
        required: true
        schema:
          type: string
        description: The managed object class name in the resource path.
      - name: id
        in: path
        required: true
        schema:
          type: string
        description: The managed object instance identifier in the resource path.
      - name: scope
        in: query
        schema:
          $ref: '#/components/schemas/Scope'
        style: form
        explode: true
        description: Optional scope parameter.
      - name: filter
        in: query
        schema:
          $ref: TS28623_ComDefs.yaml#/components/schemas

## Scenario B — retry feedback is honored

Inject a fake `validation_errors` + `validator_op_reflection`. The retry prompt should reach the LLM (we can't easily assert what it produces, but we verify the call succeeds and op_iteration_count advances).

In [12]:
# Step 6 — Retry attempt with fake validator feedback

retry_state = {
    **state,
    "op_iteration_count": 1,
    "validation_errors": [
        {
            "error_type": "correction",
            "stage": "semantic_2",
            "instruction": "Add a 404 response referencing #/components/schemas/ErrorResponse",
            "ref": f"$.paths.{path}.{method}.responses",
        }
    ],
    "validator_op_reflection": {
        "summary": "Add the missing 404 response",
        "missing_rules": ["response 404 with ErrorResponse schema"],
        "structural_issues": [],
        "semantic_priorities": ["completeness of error responses"],
    },
}
out_retry = patcher_node(retry_state)
assert out_retry["op_iteration_count"] == 2
frag_retry = out_retry["current_fragment"]
assert frag_retry["path"] == path and frag_retry["method"] == method
logger.info(
    f"Retry fragment responses: "
    f"{list(((frag_retry.get('paths') or {}).get(path) or {}).get(method, {}).get('responses', {}))}"
)

2026-05-26 16:51:20 [INFO] openapi_generator.nodes.patcher: Patcher → op 1/25 (PUT /{className}={id}) action=update attempt=2
2026-05-26 16:51:20 [WARNING] openapi_generator.rag.retriever: RAG query failed (ModuleNotFoundError: No module named 'langchain.retrievers') — returning []
2026-05-26 16:51:20 [INFO] openapi_generator.nodes.patcher: Patcher → retry: 1 correction(s) fed back to the LLM
2026-05-26 16:51:37 [INFO] openapi_generator.nodes.patcher: Patcher → produced fragment for PUT /{className}={id}: 2 path block(s), 0 new schema(s)
2026-05-26 16:51:37 [INFO] __main__: Retry fragment responses: ['200']


## Failure-mode tests (no LLM call)

When the plan is empty or the index is out of bounds, the Patcher returns an empty fragment and advances the iteration counter without invoking the LLM.

In [13]:
# Step 7 — Out-of-bounds idx → empty fragment, no LLM call

out_oob = patcher_node({"operations_plan": [], "current_op_idx": 0, "op_iteration_count": 0})
assert out_oob["current_fragment"]["paths"] == {}
assert out_oob["current_fragment"]["components"] == {}
assert out_oob["op_iteration_count"] == 1
logger.info("Out-of-bounds idx → empty fragment, counter advanced. OK.")

2026-05-26 16:51:37 [ERROR] openapi_generator.nodes.patcher: Patcher → current_op_idx=0 out of bounds (plan size=0); returning empty fragment
2026-05-26 16:51:37 [INFO] __main__: Out-of-bounds idx → empty fragment, counter advanced. OK.


In [14]:
# Step 8 — DI uniformity: sentinel retriever is invoked

calls = []
def fake_retriever(query, k=5, filters=None):
    calls.append((query, k, filters))
    return [f"FAKE CHUNK for {query!r}"]

out_di = patcher_node(state, retriever=fake_retriever)
assert calls, "Patcher must call the injected retriever"
logger.info(f"Fake retriever called {len(calls)} time(s); first query: {calls[0][0]!r}")

2026-05-26 16:51:37 [INFO] openapi_generator.nodes.patcher: Patcher → op 1/25 (PUT /{className}={id}) action=update attempt=1
2026-05-26 16:51:37 [INFO] openapi_generator.nodes.patcher: Patcher → RAG returned 1 chunk(s)
2026-05-26 16:51:59 [INFO] openapi_generator.nodes.patcher: Patcher → produced fragment for PUT /{className}={id}: 1 path block(s), 0 new schema(s)
2026-05-26 16:51:59 [INFO] __main__: Fake retriever called 1 time(s); first query: 'PUT /{className}={id} — createMOI — MnSRoot'


## Scenario C — Patcher feeds Assembler directly (skip Reflector/Validator)

Sanity check that the fragment the Patcher just produced is shaped well
enough for the Assembler to merge it into `final_openapi` without errors.
This bypasses the per-op retry loop on purpose — Reflector and Validator
are still stubs at this point.

The Assembler treats `current_fragment` as if it were a `reflected_fragment`
(same shape contract), so we forward the Patcher output under that key.

In [15]:
# Step 9 — Assembler ingests the Patcher fragment

import tempfile

from openapi_generator.nodes.assembler import assembler_node

# Patcher writes `current_fragment`; Reflector would normally pass it forward
# as `reflected_fragment`. With Reflector stubbed, we wire the Patcher output
# directly to the field the Assembler reads.
asm_state = {
    **state,
    "current_fragment": frag,
    "reflected_fragment": frag,
    "validation_errors": [],
    "op_iteration_count": 1,  # one Patcher attempt completed
    "openapi_target_path": str(Path(tempfile.mkdtemp(prefix="patcher_asm_")) / "out.yaml"),
}
asm_out = assembler_node(asm_state)

# Loop must advance and per-op scratch must be cleared.
assert asm_out["current_op_idx"] == 1
assert asm_out["op_iteration_count"] == 0
assert asm_out["current_fragment"] == {}
assert asm_out["reflected_fragment"] == {}

# The merged document must now carry the Patcher's path and schemas.
merged = asm_out["final_openapi"]
merged_paths = merged.get("paths") or {}
merged_schemas = (merged.get("components") or {}).get("schemas") or {}

# At least the fragment's path key (whatever the LLM chose) must show up.
patched_paths = list((frag.get("paths") or {}).keys())
assert any(p in merged_paths for p in patched_paths), (
    f"None of the Patcher paths {patched_paths} ended up in merged.paths "
    f"({list(merged_paths)[:5]}...)"
)

logger.info(
    f"Assembler merged the Patcher fragment: "
    f"paths={list(merged_paths)[:5]}{' ...' if len(merged_paths) > 5 else ''}; "
    f"schemas={list(merged_schemas)[:5]}{' ...' if len(merged_schemas) > 5 else ''}"
)
logger.info(f"YAML target (not written yet, more ops remain): {asm_state['openapi_target_path']}")

2026-05-26 16:51:59 [INFO] openapi_generator.nodes.assembler: Assembler → merged put /{className}={id}: +1 path block(s), +0 schema(s)
2026-05-26 16:51:59 [INFO] __main__: Assembler merged the Patcher fragment: paths=['/{className}={id}']; schemas=['CmNotificationTypes', 'SourceIndicator', 'ScopeType', 'Operation', 'Insert'] ...
2026-05-26 16:51:59 [INFO] __main__: YAML target (not written yet, more ops remain): /tmp/patcher_asm_9oim1cea/out.yaml


In [16]:
# Step 10 — Write Patcher → Assembler combined output to data/outputs/test_patcher/
#
# Two files per run, timestamped so successive runs don't clobber each other:
#   merged_<ts>.yaml      — the OpenAPI document after Assembler merged
#                           the Patcher fragment into the legacy seed.
#   fragment_<ts>.yaml    — only the Patcher's fragment, for easier diffing.

from datetime import datetime

from openapi_generator.config.paths import OUTPUTS_TEST_PATCHER_DIR

OUT_DIR = Path(OUTPUTS_TEST_PATCHER_DIR).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
merged_doc = {
    "openapi": merged.get("openapi"),
    "info": merged.get("info"),
    "paths": merged_paths,
    "components": {"schemas": merged_schemas},
}

merged_path = OUT_DIR / f"merged_{ts}.yaml"
merged_path.write_text(
    yaml.safe_dump(merged_doc, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

fragment_path = OUT_DIR / f"fragment_{ts}.yaml"
fragment_path.write_text(
    yaml.safe_dump(
        {
            "path": frag.get("path"),
            "method": frag.get("method"),
            "paths": frag.get("paths") or {},
            "components": frag.get("components") or {},
        },
        sort_keys=False, allow_unicode=True,
    ),
    encoding="utf-8",
)

logger.info(f"Wrote merged OpenAPI : {merged_path}")
logger.info(f"Wrote raw fragment   : {fragment_path}")

# Quick preview in the notebook output so you don't have to open the file
preview = merged_path.read_text(encoding="utf-8")[:2000]
print(preview)
print(f"\n...[truncated; full content in {merged_path}]")

2026-05-26 16:51:59 [INFO] __main__: Wrote merged OpenAPI : /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/data/outputs/test_patcher/merged_20260526_165159.yaml
2026-05-26 16:51:59 [INFO] __main__: Wrote raw fragment   : /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/data/outputs/test_patcher/fragment_20260526_165159.yaml


openapi: 3.0.1
info:
  title: Provisioning MnS
  version: 17.7.0
  description: OAS 3.0.1 definition of the Provisioning MnS © 2024, 3GPP Organizational
    Partners (ARIB, ATIS, CCSA, ETSI, TSDSI, TTA, TTC). All rights reserved.
paths:
  /{className}={id}:
    parameters:
    - name: className
      in: path
      required: true
      schema:
        type: string
    - name: id
      in: path
      required: true
      schema:
        type: string
    put:
      summary: Replaces a complete single resource or creates it if it does not exist
      description: With HTTP PUT a complete resource is replaced or created if it
        does not exist. The target resource is identified by the target URI.
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/Resource'
      responses:
        '200':
          description: Success case ("200 OK"). This status code shall be returned
            when th